# Optimize Bell fidelity at the Floquet NESS 


In [1]:
# --- imports ---
import os, time
from pathlib import Path

import numpy as np
from numpy import kron
from numpy.linalg import norm
from scipy.linalg import expm, eig
from scipy.optimize import minimize

import matplotlib.pyplot as plt


## 1) Physics model 


In [10]:
# =========================
# Core Liouville-space utilities (2 qubits)
# =========================

# ---- single-qubit operators ----
I2 = np.array([[1.,0.],[0.,1.]], dtype=complex)
sp = np.array([[0.,0.],[1.,0.]], dtype=complex)   # |1><0|
sm = np.array([[0.,1.],[0.,0.]], dtype=complex)   # |0><1|
sx = np.array([[0.,1.],[1.,0.]], dtype=complex)
sy = np.array([[0.,-1j],[1j,0.]], dtype=complex)
sz = np.array([[1.,0.],[0.,-1.]], dtype=complex)

# ---- 2-qubit lifted operators ----
I4  = kron(I2, I2)
sp1 = kron(sp, I2); sm1 = kron(sm, I2)
sp2 = kron(I2, sp); sm2 = kron(I2, sm)

n1_op = sp1 @ sm1
n2_op = sp2 @ sm2
p0_1  = sm1 @ sp1  # |0><0| on qubit 1 (tensored with I)
p0_2  = sm2 @ sp2

sx1 = kron(sx, I2); sy1 = kron(sy, I2); sz1 = kron(sz, I2)
sx2 = kron(I2, sx); sy2 = kron(I2, sy); sz2 = kron(I2, sz)

# ---- vec / mat in column-stacking convention ----
def vec(A: np.ndarray) -> np.ndarray:
    return np.asarray(A, dtype=complex).reshape(-1, order='F')

def mat(v: np.ndarray, d: int = 4) -> np.ndarray:
    return np.asarray(v, dtype=complex).reshape((d, d), order='F')

def trace_vec(d: int = 4) -> np.ndarray:
    return vec(np.eye(d, dtype=complex))

# ---- superoperator building blocks ----
def left_right_super(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """vec(A ρ B) = (B^T ⊗ A) vec(ρ)."""
    return kron(B.T, A)

def nh_comm_super(H_eff: np.ndarray) -> np.ndarray:
    """-i(H_eff ρ - ρ H_eff†) as a superoperator."""
    d = H_eff.shape[0]
    return -1j * (kron(np.eye(d, dtype=complex), H_eff) - kron(H_eff.conj(), np.eye(d, dtype=complex)))

def bell_state(which: str = 'psi_minus') -> np.ndarray:
    """Return |Psi±> as a 4-vector in computational basis |00>,|01>,|10>,|11>."""
    e00 = np.array([1,0,0,0], dtype=complex)
    e01 = np.array([0,1,0,0], dtype=complex)
    e10 = np.array([0,0,1,0], dtype=complex)
    if which.lower() in ['psi_minus','psiminus','-','singlet']:
        v = (e01 - e10)/np.sqrt(2)
    elif which.lower() in ['psi_plus','psiplus','+','triplet']:
        v = (e01 + e10)/np.sqrt(2)
    else:
        raise ValueError("which must be 'psi_minus' or 'psi_plus'")
    return v

def fidelity_pure(psi: np.ndarray, rho: np.ndarray) -> float:
    return float(np.real(np.vdot(psi, rho @ psi)))

def purity(rho: np.ndarray) -> float:
    return float(np.real(np.trace(rho @ rho)))

def partial_trace_rho(rho: np.ndarray, keep: int) -> np.ndarray:
    """Partial trace of a 2-qubit rho over the other qubit.
    keep=1 returns rho1, keep=2 returns rho2.
    """
    rho = np.asarray(rho, dtype=complex).reshape(2,2,2,2)  # (a,b;c,d) indices: (q1,q2;q1,q2)
    if keep == 1:
        # trace over qubit2
        return np.einsum('abcb->ac', rho)
    elif keep == 2:
        # trace over qubit1
        return np.einsum('abac->bc', rho)
    else:
        raise ValueError('keep must be 1 or 2')

def bloch_vec_single(rho1: np.ndarray) -> np.ndarray:
    bx = np.real(np.trace(rho1 @ sx))
    by = np.real(np.trace(rho1 @ sy))
    bz = np.real(np.trace(rho1 @ sz))
    return np.array([bx, by, bz], dtype=float)

def concurrence(rho: np.ndarray) -> float:
    """Wootters concurrence for 2-qubit state."""
    Y = kron(sy, sy)
    rho_tilde = Y @ rho.conj() @ Y
    R = rho @ rho_tilde
    evals = np.sort(np.real(np.linalg.eigvals(R)))[::-1]
    evals = np.maximum(evals, 0.0)
    s = np.sqrt(evals)
    C = max(0.0, float(s[0] - s[1] - s[2] - s[3]))
    return C


## 2) Fast Liouvillian assembly



In [11]:
def precompute_liouvillian_parts(g: float, alpha: float, n1: float, n2: float, q: float = 1.0):
    # Hamiltonian pieces
    H_g = g * (sp1 @ sm2 + sm1 @ sp2)
    H_delta = 0.5 * (n2_op - n1_op)

    # Non-Hermitian piece per unit gamma
    # A_nh = (1-n1)*|1><1| + n1*|0><0| on qubit1  + alpha*[(1-n2)*|1><1| + n2*|0><0|] on qubit2
    A_nh = (1.0 - n1) * n1_op + n1 * p0_1 + alpha * ((1.0 - n2) * n2_op + n2 * p0_2)
    H_gamma = -0.5j * A_nh  # so H_eff = H_g + delta*H_delta + gamma*H_gamma

    L_const = nh_comm_super(H_g)
    L_delta = nh_comm_super(H_delta)
    L_gamma_h = nh_comm_super(H_gamma)

    # Jump superoperators per unit gamma
    Jp1 = left_right_super(sp1, sm1)   # σ+ ρ σ-
    Jm1 = left_right_super(sm1, sp1)   # σ- ρ σ+
    Jp2 = left_right_super(sp2, sm2)
    Jm2 = left_right_super(sm2, sp2)

    # rates per unit gamma
    g1p = n1
    g1m = (1.0 - n1)
    g2p = alpha * n2
    g2m = alpha * (1.0 - n2)

    L_gamma_jump = q * (g1p * Jp1 + g1m * Jm1 + g2p * Jp2 + g2m * Jm2)

    L_gamma = L_gamma_h + L_gamma_jump
    return L_const, L_delta, L_gamma


def liouvillian_slice(L_const, L_delta, L_gamma, delta_val: float, gamma_val: float):
    return L_const + delta_val * L_delta + gamma_val * L_gamma


## 3) Control parameterization (Fourier)


In [12]:
# Reduced time: s = t / T in [0, 1].

def delta_from_theta(theta_delta: np.ndarray, s: np.ndarray) -> np.ndarray:
    r"""
    Fourier sine series for the detuning:
        δ(s) = Σ_{m=1..M} a_m sin(2π m s),
    where theta_delta = [a1, ..., aM].
    """
    M = len(theta_delta)
    out = np.zeros_like(s, dtype=float)
    for m in range(1, M + 1):
        out += theta_delta[m - 1] * np.sin(2 * np.pi * m * s)
    return out


def gamma_from_theta(theta_gamma: np.ndarray, s: np.ndarray) -> np.ndarray:
    r"""
    Fourier cosine series with a squared envelope:
        γ(s) = sin^2(π s) * ( b0 + Σ_{m=1..M-1} b_m cos(2π m s) )^2

    theta_gamma = [b0, b1, ..., b_{M-1}] (length M).

    Guarantees:
      • γ(s) ≥ 0
      • γ(0) = γ(1) = 0
    """
    M = len(theta_gamma)
    b0 = float(theta_gamma[0])
    bc = np.asarray(theta_gamma[1:], dtype=float)  # length M-1

    f = b0 * np.ones_like(s, dtype=float)
    for m in range(1, M):  # m = 1..M-1
        f += bc[m - 1] * np.cos(2 * np.pi * m * s)

    w = np.sin(np.pi * s) ** 2
    return w * (f ** 2)


def unpack_theta(theta: np.ndarray, M: int):
    r"""Unpack the optimization vector.

    Layout (length 2M+1):
        theta = [a1..aM, b0..b_{M-1}, logT]
    """
    theta = np.asarray(theta, dtype=float)
    theta_delta = theta[:M]
    theta_gamma = theta[M:M + M]
    logT = theta[-1]
    return theta_delta, theta_gamma, logT

## 4) Floquet map and Floquet-NESS solver

In [13]:
def floquet_map(P_list):
    Phi = np.eye(P_list[0].shape[0], dtype=complex)
    for j in range(len(P_list)-1, -1, -1):
        Phi = Phi @ P_list[j]
    return Phi

def ness_fixed_point(Phi: np.ndarray, d: int = 4):
    n = Phi.shape[0]
    c = trace_vec(d).reshape(-1, 1)
    A = np.block([[np.eye(n, dtype=complex) - Phi, c],
                  [c.T, np.zeros((1,1), dtype=complex)]])
    b = np.zeros((n+1,), dtype=complex)
    b[-1] = 1.0
    x = np.linalg.solve(A, b)
    r_star = x[:n]
    rho_star = mat(r_star, d=d)
    # numerical symmetrization
    rho_star = 0.5*(rho_star + rho_star.conj().T)
    # enforce trace exactly
    rho_star = rho_star / np.trace(rho_star)
    return r_star, rho_star

def build_propagators(theta_delta, theta_gamma, T, N,
                      L_const, L_delta, L_gamma):
    s_nodes = (np.arange(N) + 0.5) / N
    delta_s = delta_from_theta(theta_delta, s_nodes)
    gamma_s = gamma_from_theta(theta_gamma, s_nodes)
    dt = T / N
    P_list = []
    for j in range(N):
        L = liouvillian_slice(L_const, L_delta, L_gamma, delta_s[j], gamma_s[j])
        P_list.append(expm(dt * L))
    return s_nodes, delta_s, gamma_s, P_list


## 5) Objective: Bell fidelity of the Floquet fixed point


In [14]:
def evaluate_controls(theta: np.ndarray, M: int, N: int,
                      L_const, L_delta, L_gamma,
                      psi_target: np.ndarray,
                      T_bounds=(50.0, 5000.0),
                      T_penalty=0.0):
    theta_delta, theta_gamma, logT = unpack_theta(theta, M)
    T = float(np.exp(logT))
    # hard clamp (keeps exp(logT) stable even if optimizer wanders)
    T = min(max(T, T_bounds[0]), T_bounds[1])

    s_nodes, delta_s, gamma_s, P_list = build_propagators(
        theta_delta, theta_gamma, T, N, L_const, L_delta, L_gamma
    )
    Phi = floquet_map(P_list)
    r_star, rho_star = ness_fixed_point(Phi, d=4)

    F = fidelity_pure(psi_target, rho_star)
    # soft penalty (optional)
    obj = -(F - T_penalty * (T / T_bounds[1]))
    diagnostics = {
        "T": T,
        "F": F,
        "rho_star": rho_star,
        "r_star": r_star,
        "Phi": Phi,
        "P_list": P_list,
        "s_nodes": s_nodes,
        "delta_s": delta_s,
        "gamma_s": gamma_s,
    }
    return obj, diagnostics


## 6) Optimization setup


In [27]:
# --------------------------- user settings ---------------------------

# Target Bell state:
TARGET = "psi_minus"   # "psi_minus" or "psi_plus"
psi_target = bell_state(TARGET)

# Paper-like system parameters (units: ε=1, i.e. all rates are in units of ε and time is in units of 1/ε)
g = 0.01
alpha = 1.2

# Reservoir occupations (paper's "optimal" setting)
n1 = 1.0
n2 = 0.0

# Full Lindblad for a well-defined Floquet NESS
q = 1.0

# Discretization and control bandwidth
N = 64   # slices per period
M = 3    # Fourier order

# Period bounds (dimensionless, ε=1)
T_bounds = (50.0, 2500.0)

# Bounds on control parameters
# δ(s) in the paper uses Δδ≈0.04 (as a single sine). Use a wider box initially.
delta_coeff_bound = 0.2

# γ(s) = sin^2(π s) * (b0 + Σ_{m=1..M-1} b_m cos(2π m s))^2
# Since max_s sin^2(π s)=1, the typical γ scale is ~ (typical f(s))^2.
# Paper uses Δγ≈0.008 ⇒ f ≈ sqrt(0.008) ≈ 0.089. Keep a moderate search box.
gamma_coeff_bound = 0.6

# Optimizer
seed = 1
n_restarts = 8
maxiter = 120
T_penalty = 0.0   # set to 0.0 for no penalty; increase (~1e-3) only if optimizer pushes T to max

quick_mode = True
if quick_mode:
    n_restarts = 3
    maxiter = 60
    N = 48
    print("QUICK MODE: reduced N / restarts / maxiter")

rng = np.random.default_rng(seed)

# --------------------------- precompute Liouvillian parts ---------------------------
L_const, L_delta, L_gamma = precompute_liouvillian_parts(g=g, alpha=alpha, n1=n1, n2=n2, q=q)

# --------------------------- initial guess: paper loop ---------------------------
# Paper-like seed:
#   δ(s) = -Δδ sin(2π s)
#   γ(s) =  Δγ sin^2(π s)
Delta_delta_paper = 0.04
Delta_gamma_paper = 0.008

theta_delta0 = np.zeros(M)
theta_delta0[0] = -Delta_delta_paper  # sign can be flipped by the optimizer

theta_gamma0 = np.zeros(M)
theta_gamma0[0] = np.sqrt(Delta_gamma_paper)  # f(s)=const ⇒ γ(s)=Δγ sin^2(π s)

# T: paper has T*ε≈2500 (with ε=1 => T≈2500)
T0 = 2500.0
logT0 = np.log(T0)

theta0 = np.concatenate([theta_delta0, theta_gamma0, [logT0]])

# --------------------------- bounds (L-BFGS-B) ---------------------------
bounds = []
# theta_delta (M)
bounds += [(-delta_coeff_bound, delta_coeff_bound)] * M

# theta_gamma (M)
bounds += [(-gamma_coeff_bound, gamma_coeff_bound)] * M

# logT
bounds += [(np.log(T_bounds[0]), np.log(T_bounds[1]))]


# --------------------------- objective wrapper ---------------------------
def objective(theta):
    obj, _ = evaluate_controls(
        theta=theta, M=M, N=N,
        L_const=L_const, L_delta=L_delta, L_gamma=L_gamma,
        psi_target=psi_target,
        T_bounds=T_bounds,
        T_penalty=T_penalty,
    )
    return obj

QUICK MODE: reduced N / restarts / maxiter


In [28]:
# --------------------------- multi-start optimization ---------------------------

def run_one(theta_init, label="run"):
    history = {"obj": [], "F": [], "T": []}

    def cb(th):
        obj, diag = evaluate_controls(
            theta=th, M=M, N=N,
            L_const=L_const, L_delta=L_delta, L_gamma=L_gamma,
            psi_target=psi_target,
            T_bounds=T_bounds,
            T_penalty=T_penalty,
        )
        history["obj"].append(float(obj))
        history["F"].append(float(diag["F"]))
        history["T"].append(float(diag["T"]))

    res = minimize(
        objective,
        x0=theta_init,
        method="L-BFGS-B",
        bounds=bounds,
        options={"maxiter": maxiter, "ftol": 1e-10},
        callback=cb,
    )

    obj_best, diag_best = evaluate_controls(
        theta=res.x, M=M, N=N,
        L_const=L_const, L_delta=L_delta, L_gamma=L_gamma,
        psi_target=psi_target,
        T_bounds=T_bounds,
        T_penalty=T_penalty,
    )

    out = {
        "label": label,
        "res": res,
        "history": history,
        "diag": diag_best,
        "theta": res.x.copy(),
        "obj": float(obj_best),
    }
    return out


# First start: paper guess
runs = []
runs.append(run_one(theta0, label="paper_init"))

# Random restarts
for k in range(n_restarts - 1):
    th = np.empty_like(theta0)
    for i,(lo,hi) in enumerate(bounds):
        th[i] = rng.uniform(lo, hi)
    runs.append(run_one(th, label=f"rand_{k:02d}"))

# Pick best (largest fidelity => smallest negative objective)
runs_sorted = sorted(runs, key=lambda r: r["obj"])
best = runs_sorted[0]
print(f"Best run: {best['label']}")
print(f"  F*   = {best['diag']['F']:.6f}")
print(f"  T*   = {best['diag']['T']:.3f}")


Best run: rand_01
  F*   = 0.997945
  T*   = 1400.007


In [29]:
# --------------------------- diagnostics + save run ---------------------------

def floquet_spectrum(Phi: np.ndarray):
    w = np.linalg.eigvals(Phi)
    # sort by magnitude, descending
    idx = np.argsort(-np.abs(w))
    return w[idx]

def micromotion_data(diag, n_cycles_conv: int = 40):
    T = diag["T"]
    P_list = diag["P_list"]
    Phi = diag["Phi"]
    rho_star = diag["rho_star"]

    # micromotion starting at fixed point
    r = vec(rho_star)
    t_steps = np.linspace(0.0, T, len(P_list) + 1)
    F_t = []
    pur_t = []
    conc_t = []
    bx1 = []; by1 = []; bz1 = []; pur1 = []
    bx2 = []; by2 = []; bz2 = []; pur2 = []

    for j in range(len(P_list) + 1):
        rho = mat(r, d=4)
        F_t.append(fidelity_pure(psi_target, rho))
        pur_t.append(purity(rho))
        conc_t.append(concurrence(rho))

        r1 = partial_trace_rho(rho, keep=1)
        r2 = partial_trace_rho(rho, keep=2)
        b1 = bloch_vec_single(r1)
        b2 = bloch_vec_single(r2)
        bx1.append(b1[0]); by1.append(b1[1]); bz1.append(b1[2]); pur1.append(purity(r1))
        bx2.append(b2[0]); by2.append(b2[1]); bz2.append(b2[2]); pur2.append(purity(r2))

        if j < len(P_list):
            r = P_list[j] @ r

    # stroboscopic convergence from a random initial state
    def random_density_d4(rng):
        A = rng.standard_normal((4,4)) + 1j*rng.standard_normal((4,4))
        rho = A @ A.conj().T
        rho = rho / np.trace(rho)
        return rho

    rho0 = random_density_d4(rng)
    r0 = vec(rho0)
    r_star = vec(rho_star)

    strobe_idx = np.arange(n_cycles_conv + 1)
    conv_dist = []
    r = r0.copy()
    for k in strobe_idx:
        rho = mat(r, d=4)
        rho = 0.5*(rho + rho.conj().T)
        rho = rho / np.trace(rho)
        conv_dist.append(float(norm(rho - rho_star)))
        r = Phi @ r

    return {
        "t_steps": t_steps,
        "F_t": np.array(F_t, dtype=float),
        "pur_t": np.array(pur_t, dtype=float),
        "conc_t": np.array(conc_t, dtype=float),
        "bx1": np.array(bx1, dtype=float),
        "by1": np.array(by1, dtype=float),
        "bz1": np.array(bz1, dtype=float),
        "pur1": np.array(pur1, dtype=float),
        "bx2": np.array(bx2, dtype=float),
        "by2": np.array(by2, dtype=float),
        "bz2": np.array(bz2, dtype=float),
        "pur2": np.array(pur2, dtype=float),
        "strobe_idx": strobe_idx,
        "conv_dist": np.array(conv_dist, dtype=float),
    }


diag = best["diag"]
theta_best = best["theta"]
theta_delta_best, theta_gamma_best, logT_best = unpack_theta(theta_best, M)
T_best = diag["T"]

# fine grid for pretty plots
t_fine = np.linspace(0.0, T_best, 2001)
s_fine = t_fine / T_best
delta_fine = delta_from_theta(theta_delta_best, s_fine)
gamma_fine = gamma_from_theta(theta_gamma_best, s_fine)

# spectrum
eigs = floquet_spectrum(diag["Phi"])

# micromotion & convergence
micro = micromotion_data(diag, n_cycles_conv=50)

# summary scalars
rho_star = diag["rho_star"]
F_star = diag["F"]
pur_star = purity(rho_star)
conc_star = concurrence(rho_star)

print(f"Saved-state diagnostics:")
print(f"  F*(Bell)     = {F_star:.6f}")
print(f"  Purity(ρ*)   = {pur_star:.6f}")
print(f"  Concurrence  = {conc_star:.6f}")

# save
run_dir = Path('results') / 'runs'
run_dir.mkdir(parents=True, exist_ok=True)
stamp = time.strftime('%Y-%m-%d_%H%M%S')
fname = f"{stamp}_BellFloquetNESS_{TARGET}_optT_N{N}_M{M}.npz"
path = run_dir / fname

np.savez(
    path,
    # run meta
    stamp=stamp, target=TARGET,
    g=g, alpha=alpha, n1=n1, n2=n2, q=q,
    N=N, M=M, T=T_best,
    # optimized parameters
    theta_delta=theta_delta_best,
    theta_gamma=theta_gamma_best,
    # fixed point
    rho_star=rho_star,
    F_star=F_star, pur_star=pur_star, conc_star=conc_star,
    # waveforms
    t_fine=t_fine, gamma_fine=gamma_fine, delta_fine=delta_fine,
    # path samples (midpoints)
    s_nodes=diag['s_nodes'], gamma_s=diag['gamma_s'], delta_s=diag['delta_s'],
    # spectrum
    floquet_eigs=eigs,
    # histories
    hist_obj=np.array(best['history']['obj'], dtype=float),
    hist_F=np.array(best['history']['F'], dtype=float),
    hist_T=np.array(best['history']['T'], dtype=float),
    # micromotion & convergence
    **micro
)

print(f"\nRun saved to: {path}")


Saved-state diagnostics:
  F*(Bell)     = 0.997945
  Purity(ρ*)   = 0.995933
  Concurrence  = 0.995930

Run saved to: results\runs\2026-03-03_220504_BellFloquetNESS_psi_minus_optT_N48_M3.npz


## Next: Plotting notebook

Run `plot_bell_floquet_ness.ipynb` to generate publication-style figures from the latest `results/runs/*.npz`.
